# AraForge — Notebook 3: Dataset Consolidation (REVISED)
*Fixes applied:* (1) the final-architecture diagram is now a **Markdown** cell (it was a code cell that would raise `SyntaxError` if executed); (2) the master `metadata.csv` is reordered to the **paper's published schema** (`file_name, Transcript, Phonemes, Speaker_ID, Dialect, Emotion`); (3) safer dialect-folder normalization and missing-file handling.

# Environment & Drive Mount

In [ ]:
import os, shutil
import pandas as pd
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("Ready for hierarchical consolidation.")

# The Consolidation Engine

In [ ]:
# Canonical column order matching the paper's metadata-schema table.
SCHEMA = ["file_name", "Transcript", "Phonemes", "Speaker_ID", "Dialect", "Emotion"]

def get_next_project_number(speaker_dir):
    if not os.path.exists(speaker_dir): return 1
    nums = [int(f) for f in os.listdir(speaker_dir)
            if os.path.isdir(os.path.join(speaker_dir, f)) and f.isdigit()]
    return max(nums) + 1 if nums else 1

def normalize_dialect_folder(dialect):
    """'Modern Standard Arabic (MSA)' -> 'MSA'; 'Khaliji Arabic' -> 'Khaliji_Arabic'."""
    d = str(dialect)
    if '(' in d and ')' in d:
        return d.split('(')[-1].replace(')', '').strip()
    return d.strip().replace(' ', '_')

def consolidate_hierarchical(pipeline_workspace, master_dataset_dir):
    master_csv_path = os.path.join(master_dataset_dir, "metadata.csv")
    os.makedirs(master_dataset_dir, exist_ok=True)
    master_frames = []
    print(f"Consolidating into: {master_dataset_dir}")

    if os.path.exists(master_csv_path):
        print("Existing master found; appending.")
        master_frames.append(pd.read_csv(master_csv_path, sep='|', encoding='utf-8'))

    for sub_dir in os.listdir(pipeline_workspace):
        sub_dir_path = os.path.join(pipeline_workspace, sub_dir)
        if not os.path.isdir(sub_dir_path): continue
        gold_csv = os.path.join(sub_dir_path, "Ara170_Gold_Standard.csv")
        parts_dir = os.path.join(sub_dir_path, "Parts")
        if not (os.path.exists(gold_csv) and os.path.exists(parts_dir)): continue

        print(f"\nProject: {sub_dir}")
        df = pd.read_csv(gold_csv, sep='|', encoding='utf-8')
        speaker_id = str(df.iloc[0]['Speaker_ID'])
        folder_dialect = normalize_dialect_folder(df.iloc[0]['Dialect'])

        speaker_dir = os.path.join(master_dataset_dir, folder_dialect, speaker_id)
        os.makedirs(speaker_dir, exist_ok=True)
        project_num = str(get_next_project_number(speaker_dir))
        new_parts_dir = os.path.join(speaker_dir, project_num, "parts")
        os.makedirs(new_parts_dir, exist_ok=True)

        rel_paths = []
        for _, row in df.iterrows():
            clip = row['Audio Clip Number']
            src = os.path.join(parts_dir, f"{clip}.wav")
            dst = os.path.join(new_parts_dir, f"{clip}.wav")
            if os.path.exists(src):
                shutil.copy2(src, dst)
                rel_paths.append(f"{folder_dialect}/{speaker_id}/{project_num}/parts/{clip}.wav")
            else:
                print(f"  missing {src}; dropping row.")
                rel_paths.append(None)

        df['file_name'] = rel_paths
        df = df.dropna(subset=['file_name'])

        # local project metadata
        df.to_csv(os.path.join(speaker_dir, project_num, "metadata.csv"),
                  sep='|', encoding='utf-8', index=False)
        print(f"  -> {folder_dialect}/{speaker_id}/{project_num}/ ({len(df)} clips)")
        master_frames.append(df)

    if not master_frames:
        print("No data processed."); return

    print("\nUpdating master metadata.csv ...")
    master_df = pd.concat(master_frames, ignore_index=True)
    master_df = master_df.drop_duplicates(subset=['file_name'], keep='last')
    # enforce paper schema order; keep any extra columns (e.g., Confidence Score) at the end
    ordered = [c for c in SCHEMA if c in master_df.columns]
    extras  = [c for c in master_df.columns if c not in ordered]
    master_df = master_df[ordered + extras]
    master_df.to_csv(master_csv_path, sep='|', encoding='utf-8', index=False)
    print(f"Master CSV updated. Total rows: {len(master_df)}")

# Execution

In [ ]:
pipeline_workspace = input("Pipeline workspace (Notebooks 1 & 2): ").strip()
master_dataset_dir = input("Final master dataset dir (e.g., /content/drive/MyDrive/Ara170_Master): ").strip()
consolidate_hierarchical(pipeline_workspace, master_dataset_dir)

# The Resulting Architecture
```
Ara170_Master/
├── metadata.csv          # master CSV (relative paths) — schema: file_name, Transcript, Phonemes, Speaker_ID, Dialect, Emotion
├── EGY/
│   └── SPK_10492/
│       ├── 1/
│       │   ├── metadata.csv
│       │   └── parts/{1.wav, 2.wav, ...}
│       └── 2/ ...
├── LEV/ ...
├── Khaliji_Arabic/ ...
└── MSA/
    └── SPK_88291/
        └── 1/{metadata.csv, parts/...}
```